In [1]:
# ⚡ AI Energy Efficiency Monitoring - Full ML Pipeline

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, time as time_lib, warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble        import RandomForestRegressor, IsolationForest
from sklearn.metrics         import (mean_absolute_error, mean_squared_error,
                                     r2_score, mean_squared_log_error)
from sklearn.model_selection import TimeSeriesSplit
from xgboost                 import XGBRegressor
from lightgbm                import LGBMRegressor

plt.style.use("seaborn-v0_8-darkgrid")
print("Libraries loaded ✓")

Matplotlib is building the font cache; this may take a moment.


Libraries loaded ✓


In [ ]:
## Step 1 — Load & Merge Data

In [ ]:
import pathlib

# Robustly locate project root by walking up from cwd until data/train.csv is found
_cwd = pathlib.Path(os.getcwd()).resolve()
ROOT = None
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / "data" / "train.csv").exists():
        ROOT = str(candidate)
        DATA = str(candidate / "data")
        break

if ROOT is None:
    raise FileNotFoundError(
        f"Cannot locate data/train.csv — searched: {_cwd}, {_cwd.parent}, {_cwd.parent.parent}"
    )

print(f"Project root : {ROOT}")
print(f"Data folder  : {DATA}")

# pandas 2.x: read as string then convert for reliable datetime parsing
t_dtypes = {'building_id': 'int16', 'meter': 'int8', 'meter_reading': 'float32'}
w_dtypes = {'site_id': 'int8', 'air_temperature': 'float32', 'cloud_coverage': 'float32', 'dew_temperature': 'float32', 'precip_depth_1_hr': 'float32', 'sea_level_pressure': 'float32', 'wind_direction': 'float32', 'wind_speed': 'float32'}
m_dtypes = {'site_id': 'int8', 'building_id': 'int16', 'square_feet': 'int32'}
train   = pd.read_csv(f"{DATA}/train.csv", dtype=t_dtypes, nrows=2000000)
weather = pd.read_csv(f"{DATA}/weather_train.csv", dtype=w_dtypes)
meta    = pd.read_csv(f"{DATA}/building_metadata.csv", dtype=m_dtypes)

train["timestamp"]   = pd.to_datetime(train["timestamp"]).astype("datetime64[us]")
weather["timestamp"] = pd.to_datetime(weather["timestamp"]).astype("datetime64[us]")

df = train.merge(meta, on="building_id", how="left")
df = df.merge(weather, on=["site_id","timestamp"], how="left")
df = df.drop_duplicates().sort_values(["building_id","meter","timestamp"]).reset_index(drop=True)

print(f"Merged shape : {df.shape}")
print(f"Columns      : {df.columns.tolist()}")
df.head(3)


## Step 2 — Advanced Missing Value Handling (Per-Site Interpolation)

In [ ]:
weather_cont = ["air_temperature","dew_temperature","wind_speed","wind_direction",
                "cloud_coverage","sea_level_pressure","precip_depth_1_hr"]

for col in weather_cont:
    if col in df.columns:
        df[col] = (df.groupby("site_id")[col]
                     .transform(lambda s: s.interpolate(method="linear",
                                                         limit_direction="both")))

df["floor_count"] = df["floor_count"].fillna(df["floor_count"].median())
df["year_built"]  = df["year_built"].fillna(df["year_built"].median())

print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 3 — Extended Feature Engineering

### 3a — Time Features (including cyclic encoding)
Cyclic encoding prevents the model from treating hour 23 and hour 0 as far apart.


In [ ]:
df["hour"]       = df["timestamp"].dt.hour
df["day"]        = df["timestamp"].dt.day
df["month"]      = df["timestamp"].dt.month
df["weekday"]    = df["timestamp"].dt.dayofweek
df["is_weekend"] = (df["weekday"] >= 5).astype(int)
df["quarter"]    = df["timestamp"].dt.quarter

# Cyclic (sin/cos) encoding
df["hour_sin"]   = np.sin(2 * np.pi * df["hour"]  / 24)
df["hour_cos"]   = np.cos(2 * np.pi * df["hour"]  / 24)
df["month_sin"]  = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]  = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"]    = np.sin(2 * np.pi * df["weekday"] / 7)
df["dow_cos"]    = np.cos(2 * np.pi * df["weekday"] / 7)

print("Time features added ✓")

### 3b — Lag Features & Rolling Statistics

In [ ]:
grp = df.groupby(["building_id","meter"])["meter_reading"]

df["lag_1"]            = grp.shift(1)
df["lag_24"]           = grp.shift(24)
df["lag_168"]          = grp.shift(168)
df["rolling_mean_24"]  = grp.transform(lambda x: x.shift(1).rolling(24,  min_periods=1).mean())
df["rolling_mean_168"] = grp.transform(lambda x: x.shift(1).rolling(168, min_periods=1).mean())
df["rolling_std_24"]   = grp.transform(lambda x: x.shift(1).rolling(24,  min_periods=2).std())

print("Lag + rolling features added ✓")
print("NaN from lag warm-up:", df["lag_168"].isnull().sum())

### 3c — Categorical Encoding: Building Primary Use

In [ ]:
df = pd.get_dummies(df, columns=["primary_use"], prefix="use", drop_first=False)
use_cols = [c for c in df.columns if c.startswith("use_")]
print(f"One-hot columns ({len(use_cols)}): {use_cols}")

### 3d — Outlier-Aware Target Transformation

We **Winsorize** the target at the 99th percentile before log-transforming to prevent
extreme outlier buildings from dominating gradient updates.


In [ ]:
cap99 = df.groupby(["building_id","meter"])["meter_reading"].transform(
    lambda x: x.quantile(0.99))
df["meter_reading_capped"] = df["meter_reading"].clip(upper=cap99)
df["log_target"]           = np.log1p(df["meter_reading_capped"])

print("Target stats (log scale):")
print(df["log_target"].describe())

## Step 4 — Feature Selection & Chronological Train/Test Split

In [ ]:
base_features = [
    "building_id", "meter", "site_id",
    "hour", "day", "month", "weekday", "is_weekend", "quarter",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "air_temperature", "dew_temperature", "cloud_coverage",
    "wind_speed", "wind_direction", "sea_level_pressure", "precip_depth_1_hr",
    "square_feet", "floor_count", "year_built",
    "lag_1", "lag_24", "lag_168",
    "rolling_mean_24", "rolling_mean_168", "rolling_std_24",
]
feature_cols = base_features + use_cols

model_df = df[feature_cols + ["log_target","meter_reading","timestamp"]].dropna()
print(f"Modelling rows : {len(model_df):,}")
print(f"Features       : {len(feature_cols)}")

# Chronological 80/20 split
split_ts  = model_df["timestamp"].quantile(0.8)
train_df  = model_df[model_df["timestamp"] <= split_ts]
test_df   = model_df[model_df["timestamp"] >  split_ts]

X_train = train_df[feature_cols];  y_train = train_df["log_target"]
X_test  = test_df[feature_cols];   y_test  = test_df["log_target"]

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  Split: {split_ts.date()}")

## Step 5 — Multi-Model Training

Three tree-based models trained on identical features for a fair comparison.
LinearRegression is excluded — energy consumption is highly non-linear and it fails catastrophically on this dataset.


In [ ]:
models = {
    "RandomForest":  RandomForestRegressor(n_estimators=200, max_depth=15,
                                           min_samples_leaf=4, n_jobs=-1, random_state=42),
    "XGBoost":       XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=8,
                                  subsample=0.8, colsample_bytree=0.8,
                                  reg_alpha=0.1, reg_lambda=1.0,
                                  tree_method="hist", random_state=42, verbosity=0),
    "LightGBM":      LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=8,
                                   num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                                   min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                                   random_state=42, verbose=-1),
}

trained = {}
for name, m in models.items():
    print(f"  Training {name}...", end=" ", flush=True)
    m.fit(X_train, y_train)
    trained[name] = m
    print("✓")
print("\nAll models trained ✓")

## Step 6 — Cross-Validation with TimeSeriesSplit

A **TimeSeriesSplit** respects temporal ordering — never trains on future data.
This is critical for honest evaluation of energy forecasting models.


In [ ]:
tscv    = TimeSeriesSplit(n_splits=5)
cv_model = trained["LightGBM"]   # fastest model for CV

cv_maes = []
for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train)):
    Xtr, Xva = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    ytr, yva = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    cv_model.fit(Xtr, ytr)
    pred = np.clip(np.expm1(cv_model.predict(Xva)), 0, None)
    act  = np.expm1(yva)
    mae  = mean_absolute_error(act, pred)
    cv_maes.append(mae)
    print(f"  Fold {fold+1}  MAE = {mae:.2f} kWh")

print(f"\nCV MAE  mean ± std : {np.mean(cv_maes):.2f} ± {np.std(cv_maes):.2f} kWh")

# Plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, 6), cv_maes, color="#1565C0", edgecolor="white", width=0.55)
ax.axhline(np.mean(cv_maes), color="red", linestyle="--", label=f"Mean={np.mean(cv_maes):.2f}")
ax.set_title("TimeSeriesSplit Cross-Validation MAE (LightGBM)", fontsize=12)
ax.set_xlabel("Fold")
ax.set_ylabel("MAE (kWh)")
ax.legend()
plt.tight_layout()
plt.show()

## Step 7 — Hyperparameter Tuning with Optuna

**Optuna** uses Tree-of-Parzen-Estimators (TPE) — much smarter than grid search.
We optimise LightGBM (fastest) and then apply best params for the final model.
> ⏱ Reduce `n_trials` for a quick run; increase for full tuning.


In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not installed — run:  pip install optuna")
    print("Skipping hyperparameter tuning; using defaults.")

if OPTUNA_AVAILABLE:
    # Use a sample for speed (tune on 20% of training data)
    sample_size = min(100_000, len(X_train))
    Xs = X_train.sample(sample_size, random_state=42)
    ys = y_train.loc[Xs.index]

    def objective(trial):
        params = {
            "n_estimators":    trial.suggest_int("n_estimators", 100, 600, step=100),
            "learning_rate":   trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth":       trial.suggest_int("max_depth", 4, 12),
            "num_leaves":      trial.suggest_int("num_leaves", 20, 150),
            "subsample":       trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_samples":trial.suggest_int("min_child_samples", 5, 100),
            "random_state": 42, "verbose": -1,
        }
        m = LGBMRegressor(**params)
        tscv_inner = TimeSeriesSplit(n_splits=3)
        scores = []
        for tr_i, va_i in tscv_inner.split(Xs):
            m.fit(Xs.iloc[tr_i], ys.iloc[tr_i])
            pred = np.clip(np.expm1(m.predict(Xs.iloc[va_i])), 0, None)
            scores.append(mean_absolute_error(np.expm1(ys.iloc[va_i]), pred))
        return np.mean(scores)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    best_params = study.best_params
    best_params.update({"random_state": 42, "verbose": -1})
    print("\nBest params:", best_params)
    print(f"Best CV MAE : {study.best_value:.2f} kWh")

    tuned_lgb = LGBMRegressor(**best_params)
    tuned_lgb.fit(X_train, y_train)
    trained["LightGBM_Tuned"] = tuned_lgb
else:
    best_params = {}

## Step 7b — Weighted Ensemble

Blend the three tree-based models using weights inversely proportional to their CV MAE.
Ensembles reduce variance and consistently outperform any single model.


In [ ]:
# Quick per-model MAE on a validation slice to compute blend weights
val_size   = min(50_000, len(X_test))
Xv_sample  = X_test.iloc[:val_size]
yv_sample  = y_test.iloc[:val_size]

blend_maes = {}
for nm, mdl in trained.items():
    p = np.clip(np.expm1(mdl.predict(Xv_sample)), 0, None)
    blend_maes[nm] = mean_absolute_error(np.expm1(yv_sample), p)
    print(f"  {nm:20s}  val-MAE = {blend_maes[nm]:.2f}")

# Weights = 1/MAE, normalised
raw_w  = {k: 1.0 / v for k, v in blend_maes.items()}
total  = sum(raw_w.values())
weights = {k: v / total for k, v in raw_w.items()}
print("\nEnsemble weights:")
for k, w in weights.items():
    print(f"  {k:20s}  {w:.3f}")

class WeightedEnsemble:
    def __init__(self, models, weights):
        self.models  = models   # dict name->model
        self.weights = weights  # dict name->weight

    def predict(self, X):
        preds = np.zeros(len(X))
        for nm, mdl in self.models.items():
            preds += self.weights[nm] * mdl.predict(X)
        return preds

ensemble = WeightedEnsemble(trained, weights)
trained["Ensemble"] = ensemble
print("\nWeighted ensemble added to trained ✓")


## Step 8 — Full Model Evaluation (MAE · RMSE · R² · RMSLE)

In [ ]:
results = []
preds_dict = {}

for name, model in trained.items():
    log_pred  = model.predict(X_test)
    actual    = np.expm1(y_test.values)
    predicted = np.clip(np.expm1(log_pred), 0, None)

    mae   = mean_absolute_error(actual, predicted)
    rmse  = np.sqrt(mean_squared_error(actual, predicted))
    r2    = r2_score(actual, predicted)
    try:
        rmsle = np.sqrt(mean_squared_log_error(actual, predicted))
    except Exception:
        rmsle = np.nan

    results.append({"Model": name, "MAE": round(mae,2),
                    "RMSE": round(rmse,2), "R²": round(r2,4),
                    "RMSLE": round(rmsle,4)})
    preds_dict[name] = predicted

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
colors = ["#2196F3","#4CAF50","#FF9800","#9C27B0","#E53935"]
metrics = ["MAE","RMSE","R²","RMSLE"]

for ax, metric in zip(axes, metrics):
    vals = results_df.set_index("Model")[metric]
    bars = ax.bar(vals.index, vals.values,
                  color=colors[:len(vals)], edgecolor="white", width=0.5)
    ax.set_title(f"{metric}", fontsize=13)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=20)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Model Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 9 — SHAP Explainability

**SHAP (SHapley Additive exPlanations)** provides game-theory-grounded feature
importance that explains *individual* predictions, not just global averages.
This is required for any serious production AI system.


In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed — run:  pip install shap")

if SHAP_AVAILABLE:
    best_model_name = results_df.iloc[0]["Model"]
    best_model      = trained[best_model_name]

    # Use a small sample for speed
    shap_sample = X_test.sample(min(3000, len(X_test)), random_state=42)

    explainer   = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(shap_sample)

    print(f"SHAP computed for {best_model_name} on {len(shap_sample)} samples")

    # Summary plot (beeswarm)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, shap_sample, max_display=20, show=False)
    plt.title(f"SHAP Summary — {best_model_name}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Bar chart (mean absolute SHAP)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, shap_sample, plot_type="bar",
                      max_display=20, show=False)
    plt.title(f"SHAP Feature Importance — {best_model_name}", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    # Fallback: plain feature_importances_
    best_model_name = results_df.iloc[0]["Model"]
    best_model      = trained[best_model_name]
    if hasattr(best_model, "feature_importances_"):
        imp = pd.Series(best_model.feature_importances_,
                        index=feature_cols).nlargest(20).sort_values()
        fig, ax = plt.subplots(figsize=(10, 6))
        imp.plot(kind="barh", ax=ax, color="#FF6F00", edgecolor="white")
        ax.set_title(f"Feature Importance — {best_model_name}", fontsize=13)
        plt.tight_layout()
        plt.show()

## Step 10 — Anomaly Detection (IsolationForest)

Identifies buildings with **abnormal energy consumption patterns** — useful for
fault detection, energy auditing, and identifying misconfigured meters.


In [ ]:
# Aggregate hourly features per building for anomaly scoring
agg = (model_df.groupby("building_id")
       .agg(
           mean_kwh  = ("meter_reading", "mean"),
           std_kwh   = ("meter_reading", "std"),
           max_kwh   = ("meter_reading", "max"),
           zero_pct  = ("meter_reading", lambda x: (x == 0).mean()),
           n_hours   = ("meter_reading", "count"),
       ).reset_index())

agg["std_kwh"]  = agg["std_kwh"].fillna(0)
features_iso    = ["mean_kwh","std_kwh","max_kwh","zero_pct"]

iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
agg["anomaly_score"] = iso.fit_predict(agg[features_iso])
agg["anomaly"]       = (agg["anomaly_score"] == -1)

n_anom = agg["anomaly"].sum()
print(f"Anomalous buildings detected : {n_anom} / {len(agg)}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
normal  = agg[~agg["anomaly"]]
outlier = agg[ agg["anomaly"]]
ax.scatter(normal["mean_kwh"],  normal["std_kwh"],  alpha=0.4, s=15, label="Normal",   color="#4CAF50")
ax.scatter(outlier["mean_kwh"], outlier["std_kwh"], alpha=0.9, s=40, label="Anomaly",  color="#E53935", marker="X")
ax.set_xlabel("Mean kWh / hour")
ax.set_ylabel("Std kWh / hour")
ax.set_title(f"Building-Level Anomaly Detection (IsolationForest)  —  {n_anom} anomalies", fontsize=12)
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

agg[agg["anomaly"]].sort_values("mean_kwh", ascending=False).head(10)

## Step 11 — Actual vs Predicted Scatter

In [ ]:
best_name  = results_df.iloc[0]["Model"]
actual_all = np.expm1(y_test.values)
pred_all   = preds_dict[best_name]

# Sample for plot speed
n = min(10_000, len(actual_all))
rng = np.random.default_rng(42)
idx = rng.integers(0, len(actual_all), n)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(actual_all[idx], pred_all[idx], alpha=0.07, s=4, color="#1565C0")
lim = max(actual_all[idx].max(), pred_all[idx].max()) * 1.05
ax.plot([0, lim], [0, lim], "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual kWh")
ax.set_ylabel("Predicted kWh")
ax.set_title(f"Actual vs Predicted — {best_name}", fontsize=13)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.legend()
plt.tight_layout()
plt.show()

## Step 12 — 24-Hour Forecast Using Real Historical Data

In [ ]:
b_id, m_id = 0, 0
bld_df = model_df[(model_df["building_id"] == b_id) & (model_df["meter"] == m_id)].copy()

if len(bld_df) >= 48:
    history  = bld_df.iloc[-48:-24]
    future   = bld_df.iloc[-24:]

    best_m   = trained[best_name]
    lgb_m    = trained.get("LightGBM_Tuned", trained["LightGBM"])

    best_pred = np.clip(np.expm1(best_m.predict(history[feature_cols])), 0, None)
    lgb_pred  = np.clip(np.expm1(lgb_m.predict(history[feature_cols])), 0, None)
    actual_kw = np.expm1(future["log_target"].values)
    hours     = future["timestamp"].values

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(hours, actual_kw,  label="Actual",              color="#1E88E5", linewidth=2)
    ax.plot(history["timestamp"].values, best_pred,
            label=f"{best_name} Forecast",    color="#E53935", linestyle="--", linewidth=2)
    ax.plot(history["timestamp"].values, lgb_pred,
            label="LightGBM Forecast", color="#43A047", linestyle=":",  linewidth=2)
    ax.set_title(f"24-Hour Forecast — Building {b_id}  ·  {['Electricity','Chilled Water','Steam','Hot Water'][m_id]}",
                 fontsize=13)
    ax.set_ylabel("kWh")
    ax.legend()
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    mae_24 = mean_absolute_error(actual_kw[:len(best_pred)], best_pred)
    print(f"24-hr forecast MAE ({best_name}): {mae_24:.2f} kWh")
else:
    print("Not enough rows — try a different building_id / meter")

## Step 13 — Real-Time Monitoring Simulation

In [ ]:
from IPython.display import clear_output

sim_df = (test_df[(test_df["building_id"] == b_id) & (test_df["meter"] == m_id)]
          .head(30))

live_actuals, live_preds, live_errors = [], [], []
best_m = trained[best_name]

print("=== Real-Time Simulation (30 steps) ===\n")
print(f"{'Step':>4} | {'Actual kWh':>10} | {'Predicted kWh':>13} | {'Error':>8}")
print("-" * 48)

for i, (idx, row) in enumerate(sim_df.iterrows()):
    act  = row["meter_reading"]
    pred = float(np.clip(np.expm1(best_m.predict(
               row[feature_cols].values.reshape(1, -1))[0]), 0, None))
    err  = abs(act - pred)
    live_actuals.append(act)
    live_preds.append(pred)
    live_errors.append(err)
    print(f"{i+1:>4} | {act:>10.2f} | {pred:>13.2f} | {err:>8.2f}")
    time_lib.sleep(0.04)

print("\n=== Complete ===")
print(f"Rolling MAE  : {np.mean(live_errors):.2f} kWh")
print(f"Rolling RMSE : {np.sqrt(np.mean(np.square(live_errors))):.2f} kWh")

## Step 14 — Save Best Model & All Artifacts

In [ ]:
os.makedirs(f"{ROOT}/models", exist_ok=True)

best_name  = results_df.iloc[0]["Model"]
best_model = trained[best_name]

joblib.dump(best_model,  f"{ROOT}/models/{best_name.lower().replace(' ','_')}_model.pkl")
joblib.dump(feature_cols,f"{ROOT}/models/feature_cols.pkl")
joblib.dump(results_df,  f"{ROOT}/models/model_results.pkl")

if OPTUNA_AVAILABLE and best_params:
    joblib.dump(best_params, f"{ROOT}/models/best_hyperparams.pkl")

# Save anomaly detector
joblib.dump(iso, f"{ROOT}/models/anomaly_detector.pkl")
joblib.dump(agg, f"{ROOT}/models/building_profiles.pkl")

print(f"Best model saved  : {best_name}")
print(f"Model directory   : {ROOT}/models/")
print(f"Artifacts saved   : feature_cols, model_results, anomaly_detector, building_profiles")
print(f"\nFinal Leaderboard:\n{results_df.to_string(index=False)}")